In [1]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma

from scripts.schemas import ChunkMetadata, RankingKeywords

In [2]:
# =============================================================================
# Configuration
# =============================================================================

# ChromaDB Configuration (from PageRAG - Data Ingestion)
CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "legal_docs"
EMBEDDING_MODEL = "nomic-embed-text"
BASE_URL = "http://localhost:11434"

LLM_MODEL = "qwen3:latest"

In [3]:
# ollama pull nomic-embed_text
embeddings= OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=BASE_URL, num_ctx=8192)

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)

llm = ChatOllama(model=LLM_MODEL, base_url=BASE_URL)

llm.invoke("Hello, LangGraph and Ollama!")

AIMessage(content="Hello! It's great to see you excited about **LangGraph** and **Ollama**! 🌟\n\n- **LangGraph** is a framework for building and deploying language models, often used for tasks like text generation, reasoning, and structured data processing. It emphasizes modularity and scalability for complex workflows.\n- **Ollama** is a tool for running large language models (LLMs) locally, allowing you to manage, run, and interact with models like Llama, Mistral, and more without relying on external APIs.\n\nIf you're exploring how these tools work together, have questions about their integration, or need guidance on building workflows, feel free to ask! I'm here to help you dive deeper into their capabilities. 😊 What would you like to know next?", additional_kwargs={}, response_metadata={'model': 'qwen3:latest', 'created_at': '2026-05-14T10:01:12.109913Z', 'done': True, 'done_reason': 'stop', 'total_duration': 86266684142, 'load_duration': 7294855936, 'prompt_eval_count': 19, 'prom

In [ ]:
def extract_filters(user_query:str):

    llm_structured = llm.with_structured_output(ChunkMetadata)

    prompt = f"""Extract metadata filters from the query. Return None for fields not mentioned.

                USER QUERY: {user_query}

                CLAW TYPE  MAPPINGS:
                - Luật dân sự/ civil law -> civil law

                EXAMPLES:
                "Luật dân sự 2015" -> {{"law_type": "civil", "publish_year": 2015}}

                Extract metadata:
                """
    
    metadata = llm_structured.invoke(prompt)
    filters = metadata.model_dump(exclude_none=True)

    return filters